# BÀI TẬP: FINE-TUNING MÔ HÌNH CLIP TRÊN DỮ LIỆU HÌNH ẢNH - VĂN BẢN TIẾNG VIỆT
**Phương pháp:** Multilingual CLIP (`xlm-roberta-base-ViT-B-32`) + PEFT LoRA  
**Tập dữ liệu:** KTVIC (`leo040802/ktvic-dataset`) & UIT-VIC (`leo040802/uitvic-dataset`)  
**Môi trường đề xuất:** Google Colab / Kaggle (GPU T4/P100/A100)

---
## Mục tiêu:
1. Tiền xử lý và hợp nhất hai tập dữ liệu hình ảnh - văn bản tiếng Việt từ định dạng COCO JSON.
2. Fine-tune mô hình Multilingual CLIP sử dụng kỹ thuật **LoRA (PEFT)** trên Text Encoder.
3. Triển khai hàm mất mát **Symmetric InfoNCE Loss**.
4. Đánh giá định lượng qua các chỉ số **Recall@K ($R@1, R@5, R@10$)** và **MRR (Mean Reciprocal Rank)** trước và sau fine-tuning.
5. Trực quan hóa định tính kết quả tìm kiếm Text-to-Image và phân tích các Failure Cases.


## Phần 1: Tải và Tiền xử lý Dữ liệu (KTVIC & UIT-VIC)


In [4]:
!pip install -q open_clip_torch transformers peft kagglehub datasets accelerate torchvision pillow matplotlib seaborn pandas tqdm scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00


In [5]:
import os
import glob
import json
import pandas as pd
from PIL import Image
import kagglehub

print("--- Downloading Datasets from Kaggle ---")
ktvic_path = kagglehub.dataset_download("leo040802/ktvic-dataset")
uitvic_path = kagglehub.dataset_download("leo040802/uitvic-dataset")

print(f"KTVIC downloaded to: {ktvic_path}")
print(f"UIT-VIC downloaded to: {uitvic_path}")

def load_coco_dataset(base_path, dataset_name):
    pairs = []
    json_files = glob.glob(os.path.join(base_path, "**", "*.json"), recursive=True)

    all_image_paths = {}
    for root, _, files in os.walk(base_path):
        for f in files:
            if f.lower().endswith(('.jpg', '.png', '.jpeg', '.webp')):
                all_image_paths[f] = os.path.join(root, f)

    for json_f in json_files:
        try:
            with open(json_f, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, dict) and "images" in data and "annotations" in data:
                id_to_filename = {}
                for img in data["images"]:
                    fname = img.get("filename") or img.get("file_name")
                    if fname:
                        id_to_filename[img["id"]] = fname

                for ann in data["annotations"]:
                    img_id = ann.get("image_id")
                    caption = ann.get("caption") or ann.get("segment_caption")
                    if img_id in id_to_filename and caption:
                        fname = id_to_filename[img_id]
                        img_p = all_image_paths.get(fname)
                        if img_p and os.path.exists(img_p):
                            pairs.append({
                                "image_path": img_p,
                                "caption": caption.strip(),
                                "source": dataset_name
                            })
        except Exception as e:
            print(f"Lỗi khi đọc file {json_f}: {e}")
    return pairs

ktvic_pairs = load_coco_dataset(ktvic_path, "KTVIC")
uitvic_pairs = load_coco_dataset(uitvic_path, "UIT-VIC")

all_pairs = ktvic_pairs + uitvic_pairs
print(f"Tổng số cặp (Ảnh - Văn bản) tìm thấy: {len(all_pairs)} (KTVIC: {len(ktvic_pairs)}, UIT-VIC: {len(uitvic_pairs)})")

df_data = pd.DataFrame(all_pairs)
df_data = df_data.dropna().drop_duplicates(subset=["caption"]).reset_index(drop=True)
print(f"Số lượng dữ liệu sau khi loại bỏ trùng lặp: {len(df_data)}")
df_data.head()


--- Downloading Datasets from Kaggle ---
Using Colab cache for faster access to the 'ktvic-dataset' dataset.
Using Colab cache for faster access to the 'uitvic-dataset' dataset.
KTVIC downloaded to: /kaggle/input/ktvic-dataset
UIT-VIC downloaded to: /kaggle/input/uitvic-dataset
Tổng số cặp (Ảnh - Văn bản) tìm thấy: 36271 (KTVIC: 21635, UIT-VIC: 14636)
Số lượng dữ liệu sau khi loại bỏ trùng lặp: 30192


,image_path,caption,source
0,/kaggle/input/ktvic-dataset/ktvic_dataset/publ...,đây là khung cảnh xuất hiện ở phía trước một c...,KTVIC
1,/kaggle/input/ktvic-dataset/ktvic_dataset/publ...,có một căn nhà cao tầng xuất hiện ở trong bức ảnh,KTVIC
2,/kaggle/input/ktvic-dataset/ktvic_dataset/publ...,ở trong bức ảnh có sự xuất hiện của một căn nh...,KTVIC
3,/kaggle/input/ktvic-dataset/ktvic_dataset/publ...,có một chiếc xe máy xuất hiện ở trong căn nhà,KTVIC
4,/kaggle/input/ktvic-dataset/ktvic_dataset/publ...,đây là bức ảnh chụp ở phía trước của một căn n...,KTVIC


In [6]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

# Chia tập dữ liệu 80% Train / 10% Val / 10% Test
train_df, test_df = train_test_split(df_data, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42)

print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)} | Test samples: {len(test_df)}")

class ImageTextDataset(Dataset):
    def __init__(self, df, preprocess, tokenizer):
        self.df = df.reset_index(drop=True)
        self.preprocess = preprocess
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row["image_path"]
        caption = row["caption"]

        image = Image.open(image_path).convert("RGB")
        processed_image = self.preprocess(image)
        tokens = self.tokenizer(caption)[0]

        return {
            "image": processed_image,
            "text": tokens,
            "caption": caption,
            "image_path": image_path
        }


Train samples: 24153 | Val samples: 3019 | Test samples: 3020


## Phần 2: Khởi tạo Mô hình Multilingual CLIP & Tích hợp PEFT LoRA


In [7]:
import open_clip
from peft import get_peft_model, LoraConfig
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

MODEL_NAME = "xlm-roberta-base-ViT-B-32"
PRETRAINED_TAG = "laion5b_s13b_b90k"

model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED_TAG)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query", "value", "key", "out_proj"],
    lora_dropout=0.1,
    bias="none"
)

model.text.transformer = get_peft_model(model.text.transformer, lora_config)
model.to(device)

print("--- Thông số trainable sau khi áp dụng LoRA ---")
model.text.transformer.print_trainable_parameters()


Using device: cuda


open_clip_pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.46GB            

open_clip_pytorch_model.bin: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [8]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Đã đăng nhập Hugging Face thành công!")
except Exception as e:
    print(f"Không tìm thấy Secret 'HF_TOKEN' hoặc có lỗi xảy ra: {e}")
    print("Bạn vẫn có thể tiếp tục, nhưng có thể bị giới hạn tốc độ tải.")

Không tìm thấy Secret 'HF_TOKEN' hoặc có lỗi xảy ra: Secret HF_TOKEN does not exist.
Bạn vẫn có thể tiếp tục, nhưng có thể bị giới hạn tốc độ tải.


### Bước 1: Đánh giá Baseline Zero-shot
Tôi sẽ khởi chạy việc trích xuất embedding từ mô hình gốc chưa qua huấn luyện trên tập dữ liệu tiếng Việt để làm mốc so sánh.

In [12]:
# Bước 1: Định nghĩa các hàm hỗ trợ và chạy Zero-shot Baseline
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

def extract_all_embeddings(model, dataloader, device):
    model.eval()
    all_img_embeds = []
    all_text_embeds = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Trích xuất Embeddings"):
            images, texts = batch["image"].to(device), batch["text"].to(device)
            img_feats = model.encode_image(images)
            text_feats = model.encode_text(texts)
            all_img_embeds.append(F.normalize(img_feats, p=2, dim=-1).cpu())
            all_text_embeds.append(F.normalize(text_feats, p=2, dim=-1).cpu())
    return torch.cat(all_img_embeds, dim=0).numpy(), torch.cat(all_text_embeds, dim=0).numpy()

def compute_metrics(image_embeds, text_embeds, k_values=[1, 5, 10]):
    image_embeds = F.normalize(torch.tensor(image_embeds), p=2, dim=-1).numpy()
    text_embeds = F.normalize(torch.tensor(text_embeds), p=2, dim=-1).numpy()
    sim_t2i = text_embeds @ image_embeds.T
    n = sim_t2i.shape[0]
    t2i_ranks = []
    for i in range(n):
        sorted_idx = np.argsort(-sim_t2i[i])
        rank = np.where(sorted_idx == i)[0][0] + 1
        t2i_ranks.append(rank)
    metrics = {f"T2I_R@{k}": np.mean([1.0 if r <= k else 0.0 for r in t2i_ranks]) for k in k_values}
    metrics["T2I_MRR"] = np.mean([1.0 / r for r in t2i_ranks])
    return metrics

print("--- Đang chạy Zero-Shot Baseline ---")
test_dataset = ImageTextDataset(test_df, preprocess, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)
baseline_model, _, _ = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED_TAG)
baseline_model.to(device)

zero_img_embeds, zero_text_embeds = extract_all_embeddings(baseline_model, test_loader, device)
baseline_metrics = compute_metrics(zero_img_embeds, zero_text_embeds)

print("=== KẾT QUẢ ZERO-SHOT BASELINE ===")
for metric, val in baseline_metrics.items():
    print(f"{metric}: {val:.4f}")

--- Đang chạy Zero-Shot Baseline ---


Trích xuất Embeddings: 100%|██████████| 48/48 [01:11<00:00,  1.50s/it]


=== KẾT QUẢ ZERO-SHOT BASELINE ===
T2I_R@1: 0.1252
T2I_R@5: 0.2940
T2I_R@10: 0.3811
T2I_MRR: 0.2121


### Bước 2: Bắt đầu Fine-tuning với LoRA
Bây giờ, tôi sẽ chạy mã huấn luyện để cập nhật trọng số của Text Encoder bằng kỹ thuật LoRA.

In [14]:
# Bước 2: Định nghĩa Loss và thực hiện Fine-tuning
class SymmetricInfoNCELoss(nn.Module):
    def __init__(self, logit_scale=100.0):
        super().__init__()
        self.logit_scale = logit_scale
        self.cross_entropy = nn.CrossEntropyLoss()
    def forward(self, image_features, text_features):
        image_features = F.normalize(image_features, p=2, dim=-1)
        text_features = F.normalize(text_features, p=2, dim=-1)
        logits_per_image = self.logit_scale * (image_features @ text_features.T)
        logits_per_text = logits_per_image.T
        labels = torch.arange(len(image_features), device=image_features.device)
        return (self.cross_entropy(logits_per_image, labels) + self.cross_entropy(logits_per_text, labels)) / 2.0

import torch.optim as optim
train_loader = DataLoader(ImageTextDataset(train_df, preprocess, tokenizer), batch_size=32, shuffle=True)
val_loader = DataLoader(ImageTextDataset(val_df, preprocess, tokenizer), batch_size=64, shuffle=False)

# Đảm bảo toàn bộ mô hình (bao gồm các lớp LoRA mới thêm) đều ở trên GPU
model.to(device)

criterion = SymmetricInfoNCELoss(logit_scale=100.0)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

EPOCHS = 3
best_val_r1 = 0.0
for epoch in range(EPOCHS):
    model.train()
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        images, texts = batch["image"].to(device), batch["text"].to(device)
        optimizer.zero_grad()
        img_feats, text_feats = model.encode_image(images), model.encode_text(texts)
        loss = criterion(img_feats, text_feats)
        loss.backward()
        optimizer.step()

    # Đánh giá sau mỗi epoch
    val_img, val_txt = extract_all_embeddings(model, val_loader, device)
    val_metrics = compute_metrics(val_img, val_txt)
    print(f"Epoch {epoch+1} - Val R@1: {val_metrics['T2I_R@1']:.4f}")

    if val_metrics['T2I_R@1'] > best_val_r1:
        best_val_r1 = val_metrics['T2I_R@1']
        os.makedirs("checkpoints", exist_ok=True)
        torch.save(model.state_dict(), "checkpoints/best_clip_lora.pt")

Trích xuất Embeddings: 100%|██████████| 48/48 [01:01<00:00,  1.28s/it]


Epoch 1 - Val R@1: 0.0944


Trích xuất Embeddings: 100%|██████████| 48/48 [01:01<00:00,  1.28s/it]


Epoch 2 - Val R@1: 0.1113


Trích xuất Embeddings: 100%|██████████| 48/48 [01:01<00:00,  1.29s/it]


Epoch 3 - Val R@1: 0.1232


### Bước 3: So sánh hiệu năng
Cuối cùng, tôi sẽ nạp mô hình tốt nhất và hiển thị bảng so sánh giữa trước và sau khi fine-tune.

In [15]:
# Tải trọng số tốt nhất đã lưu và đánh giá trên tập Test
if os.path.exists("checkpoints/best_clip_lora.pt"):
    model.load_state_dict(torch.load("checkpoints/best_clip_lora.pt"))
    print("--- Đã nạp thành công checkpoint tốt nhất ---")

    ft_img, ft_txt = extract_all_embeddings(model, test_loader, device)
    finetuned_metrics = compute_metrics(ft_img, ft_txt)

    comparison_df = pd.DataFrame([
        {"Trạng thái": "Trước Fine-tune (Zero-shot)", **baseline_metrics},
        {"Trạng thái": "Sau Fine-tune (LoRA)", **finetuned_metrics}
    ])
    display(comparison_df)
else:
    print("Lỗi: Không tìm thấy file 'checkpoints/best_clip_lora.pt'. Vui lòng kiểm tra lại quá trình huấn luyện.")

--- Đã nạp thành công checkpoint tốt nhất ---


Trích xuất Embeddings: 100%|██████████| 48/48 [01:02<00:00,  1.31s/it]


,Trạng thái,T2I_R@1,T2I_R@5,T2I_R@10,T2I_MRR
0,Trước Fine-tune (Zero-shot),0.125166,0.294040,0.381126,0.212059
1,Sau Fine-tune (LoRA),0.131126,0.337417,0.461258,0.236052


In [16]:
class SymmetricInfoNCELoss(nn.Module):
    def __init__(self, logit_scale=100.0):
        super().__init__()
        self.logit_scale = logit_scale
        self.cross_entropy = nn.CrossEntropyLoss()

    def forward(self, image_features, text_features):
        image_features = F.normalize(image_features, p=2, dim=-1)
        text_features = F.normalize(text_features, p=2, dim=-1)

        logits_per_image = self.logit_scale * (image_features @ text_features.T)
        logits_per_text = logits_per_image.T

        labels = torch.arange(len(image_features), device=image_features.device)

        loss_i2t = self.cross_entropy(logits_per_image, labels)
        loss_t2i = self.cross_entropy(logits_per_text, labels)
        return (loss_i2t + loss_t2i) / 2.0

import numpy as np

def compute_metrics(image_embeds, text_embeds, k_values=[1, 5, 10]):
    image_embeds = F.normalize(torch.tensor(image_embeds), p=2, dim=-1).numpy()
    text_embeds = F.normalize(torch.tensor(text_embeds), p=2, dim=-1).numpy()

    sim_t2i = text_embeds @ image_embeds.T
    n = sim_t2i.shape[0]

    t2i_ranks = []
    t2i_recalls = {k: 0.0 for k in k_values}
    for i in range(n):
        sorted_idx = np.argsort(-sim_t2i[i])
        rank = np.where(sorted_idx == i)[0][0] + 1
        t2i_ranks.append(rank)
        for k in k_values:
            if rank <= k:
                t2i_recalls[k] += 1.0

    t2i_mrr = np.mean([1.0 / r for r in t2i_ranks])
    metrics = {f"T2I_R@{k}": t2i_recalls[k] / n for k in k_values}
    metrics["T2I_MRR"] = t2i_mrr

    sim_i2t = sim_t2i.T
    i2t_ranks = []
    i2t_recalls = {k: 0.0 for k in k_values}
    for i in range(n):
        sorted_idx = np.argsort(-sim_i2t[i])
        rank = np.where(sorted_idx == i)[0][0] + 1
        i2t_ranks.append(rank)
        for k in k_values:
            if rank <= k:
                i2t_recalls[k] += 1.0

    i2t_mrr = np.mean([1.0 / r for r in i2t_ranks])
    for k in k_values:
        metrics[f"I2T_R@{k}"] = i2t_recalls[k] / n
    metrics["I2T_MRR"] = i2t_mrr

    return metrics


## Phần 3: Đánh giá Baseline Zero-shot (Trước khi Fine-tune)


In [17]:
from tqdm import tqdm

def extract_all_embeddings(model, dataloader, device):
    model.eval()
    all_img_embeds = []
    all_text_embeds = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Trích xuất Embeddings"):
            images = batch["image"].to(device)
            texts = batch["text"].to(device)

            img_feats = model.encode_image(images)
            text_feats = model.encode_text(texts)

            img_feats = F.normalize(img_feats, p=2, dim=-1)
            text_feats = F.normalize(text_feats, p=2, dim=-1)

            all_img_embeds.append(img_feats.cpu())
            all_text_embeds.append(text_feats.cpu())

    return torch.cat(all_img_embeds, dim=0).numpy(), torch.cat(all_text_embeds, dim=0).numpy()

test_dataset = ImageTextDataset(test_df, preprocess, tokenizer)
# Dùng num_workers=0 để tránh lỗi multiprocessing khi chạy trong Jupyter Notebook
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

print("--- Trích xuất Zero-Shot Baseline Embeddings trên Test Set ---")
baseline_model, _, _ = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED_TAG)
baseline_model.to(device)

zero_img_embeds, zero_text_embeds = extract_all_embeddings(baseline_model, test_loader, device)

baseline_metrics = compute_metrics(zero_img_embeds, zero_text_embeds)
print("=== KẾT QUẢ ZERO-SHOT BASELINE (TRƯỚC FINE-TUNE) ===")
for metric, val in baseline_metrics.items():
    print(f"{metric}: {val:.4f}")


--- Trích xuất Zero-Shot Baseline Embeddings trên Test Set ---


Trích xuất Embeddings: 100%|██████████| 48/48 [01:06<00:00,  1.38s/it]


=== KẾT QUẢ ZERO-SHOT BASELINE (TRƯỚC FINE-TUNE) ===
T2I_R@1: 0.1252
T2I_R@5: 0.2940
T2I_R@10: 0.3811
T2I_MRR: 0.2121
I2T_R@1: 0.1414
I2T_R@5: 0.3169
I2T_R@10: 0.4159
I2T_MRR: 0.2327


## Phần 4: Quy trình Huấn luyện Fine-Tuning với LoRA & Optimizer


In [ ]:
import torch.optim as optim

train_dataset = ImageTextDataset(train_df, preprocess, tokenizer)
val_dataset = ImageTextDataset(val_df, preprocess, tokenizer)

# Dùng num_workers=0 để tương thích tốt với môi trường Notebook
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

criterion = SymmetricInfoNCELoss(logit_scale=100.0)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

EPOCHS = 5
history = []
best_val_r1 = 0.0

print("--- Bắt đầu Fine-tuning Mô hình CLIP ---")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images = batch["image"].to(device)
        texts = batch["text"].to(device)

        optimizer.zero_grad()

        img_feats = model.encode_image(images)
        text_feats = model.encode_text(texts)

        loss = criterion(img_feats, text_feats)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # Validation Evaluation
    val_img_embeds, val_text_embeds = extract_all_embeddings(model, val_loader, device)
    val_metrics = compute_metrics(val_img_embeds, val_text_embeds)

    val_r1 = val_metrics["T2I_R@1"]
    print(f"Epoch {epoch+1} - Loss: {avg_train_loss:.4f} | Val T2I R@1: {val_r1:.4f} | Val T2I R@5: {val_metrics['T2I_R@5']:.4f} | Val T2I MRR: {val_metrics['T2I_MRR']:.4f}")

    history.append({
        "epoch": epoch + 1,
        "loss": avg_train_loss,
        **val_metrics
    })

    # Lưu Best Model Checkpoint
    if val_r1 > best_val_r1:
        best_val_r1 = val_r1
        os.makedirs("checkpoints", exist_ok=True)
        torch.save(model.state_dict(), "checkpoints/best_clip_lora.pt")
        print("--> Đã lưu Best Model Checkpoint tại 'checkpoints/best_clip_lora.pt'!")


--- Bắt đầu Fine-tuning Mô hình CLIP ---


Trích xuất Embeddings: 100%|██████████| 48/48 [01:02<00:00,  1.30s/it]


Epoch 1 - Loss: 0.5839 | Val T2I R@1: 0.1335 | Val T2I R@5: 0.3511 | Val T2I MRR: 0.2433
--> Đã lưu Best Model Checkpoint tại 'checkpoints/best_clip_lora.pt'!


Trích xuất Embeddings: 100%|██████████| 48/48 [01:02<00:00,  1.29s/it]


Epoch 2 - Loss: 0.4637 | Val T2I R@1: 0.1318 | Val T2I R@5: 0.3548 | Val T2I MRR: 0.2427


Epoch 3/5:  62%|██████▏   | 469/755 [09:15<05:47,  1.22s/it]

## Phần 5: Đánh giá Định lượng & So sánh Kết quả (Quantitative Evaluation)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Nạp Best Model Checkpoint và đánh giá trên Test Set
model.load_state_dict(torch.load("checkpoints/best_clip_lora.pt"))
print("--- Trích xuất Fine-Tuned Model Embeddings trên Test Set ---")
ft_img_embeds, ft_text_embeds = extract_all_embeddings(model, test_loader, device)

finetuned_metrics = compute_metrics(ft_img_embeds, ft_text_embeds)

# Bảng so sánh Before vs After
comparison_df = pd.DataFrame([
    {"Trạng thái": "Trước Fine-tune (Zero-shot)", **baseline_metrics},
    {"Trạng thái": "Sau Fine-tune (LoRA)", **finetuned_metrics}
])

print("=== BẢNG SO SÁNH HIỆU NĂNG TRÊN TẬP TEST ===")
display(comparison_df)

# Vẽ biểu đồ thay đổi chỉ số qua từng Epoch
history_df = pd.DataFrame(history)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_df["epoch"], history_df["T2I_R@1"], label="T2I R@1", marker='o')
plt.plot(history_df["epoch"], history_df["T2I_R@5"], label="T2I R@5", marker='s')
plt.plot(history_df["epoch"], history_df["T2I_R@10"], label="T2I R@10", marker='^')
plt.title("Text-to-Image Recall@K qua từng Epoch")
plt.xlabel("Epoch")
plt.ylabel("Recall Score")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_df["epoch"], history_df["T2I_MRR"], label="T2I MRR", color='purple', marker='d')
plt.plot(history_df["epoch"], history_df["I2T_MRR"], label="I2T MRR", color='orange', marker='h')
plt.title("Mean Reciprocal Rank (MRR) qua từng Epoch")
plt.xlabel("Epoch")
plt.ylabel("MRR Score")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


## Phần 6: Đánh giá Định tính & Trực quan hóa Top-5 Results


In [ ]:
sample_queries = [
    "Một cô gái mặc áo dài truyền thống đứng bên hồ Hoàn Kiếm",
    "Tách cà phê sữa đá trên bàn gỗ ngoài quán vỉa hè",
    "Bát phở bò nóng hổi kèm quẩy và hành lá",
    "Người lái xe xích lô chở du khách trên phố cổ Hà Nội",
    "Cảnh tấp nập mua bán trước cổng chợ Bến Thành"
]

def visualize_top_k(eval_model, queries, dataset, k=5, title_prefix=""):
    eval_model.eval()
    all_images = [Image.open(row["image_path"]).convert("RGB") for _, row in dataset.df.iterrows()]

    img_tensors = torch.stack([preprocess(img) for img in all_images]).to(device)
    with torch.no_grad():
        img_embeddings = eval_model.encode_image(img_tensors)
        img_embeddings = F.normalize(img_embeddings, p=2, dim=-1)

        text_tokens = tokenizer(queries).to(device)
        text_embeddings = eval_model.encode_text(text_tokens)
        text_embeddings = F.normalize(text_embeddings, p=2, dim=-1)

        sim_matrix = (text_embeddings @ img_embeddings.T).cpu().numpy()

    for idx, query in enumerate(queries):
        scores = sim_matrix[idx]
        top_k_indices = np.argsort(-scores)[:k]

        fig, axes = plt.subplots(1, k, figsize=(18, 4))
        fig.suptitle(f"{title_prefix} Query: '{query}'", fontsize=14, fontweight='bold')

        for rank, img_idx in enumerate(top_k_indices):
            img = all_images[img_idx]
            score = scores[img_idx]

            axes[rank].imshow(img)
            axes[rank].set_title(f"Top-{rank+1} (Score: {score:.3f})")
            axes[rank].axis('off')

        plt.tight_layout()
        plt.show()

print("--- Trực quan hóa kết quả tìm kiếm TRƯỚC khi Fine-tune ---")
visualize_top_k(baseline_model, sample_queries[:2], test_dataset, k=5, title_prefix="[Zero-shot]")

print("--- Trực quan hóa kết quả tìm kiếm SAU khi Fine-tune ---")
visualize_top_k(model, sample_queries[:2], test_dataset, k=5, title_prefix="[Fine-tuned]")


## Báo cáo Phân tích Failure Cases (Các trường hợp thất bại)

### Nguyên nhân mô hình đưa ra kết quả chưa chính xác:
1. **Dữ liệu nhiễu và độ chi tiết câu mô tả:** Một số câu mô tả trong dataset COCO ngắn hoặc không đề cập tới các chi tiết phụ trong ảnh (ví dụ: *ảnh chụp bối cảnh lớn nhưng caption chỉ nêu đối tượng trung tâm*).
2. **Khái niệm đặc thù văn hóa Việt Nam:** Các thuật ngữ như *xe xích lô*, *phở bò*, *nón lá*, *chợ Bến Thành* có tần suất xuất hiện vừa phải trong KTVIC & UIT-VIC, cần thêm số lượng epoch hoặc tiếp tục mở rộng tập dữ liệu để nâng cao độ chính xác biểu diễn không gian ngữ nghĩa.
3. **Giới hạn Resolution:** Mô hình CLIP mặc định ($224 	imes 224$) thu nhỏ hình ảnh làm mất chi tiết các biển hiệu tiếng Việt hoặc chi tiết nhỏ trong các tấm ảnh toàn cảnh.
